# Loss analysis: reviewer evidence

Present the canonical full-cohort runner outputs, including raw density-processing exclusions. Run `.venv/bin/python paper_analysis/run_review.py` first. All exports remain repository-local.


In [1]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Works from paper_analysis/, examples/, the repository, or the Snow workspace.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "SnowPyt-MechParams"]
repo_root = next(p for p in candidates if (p / "src/snowpyt_mechparams").is_dir())
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "examples"))
sys.path.insert(0, str(repo_root / "paper_analysis"))

from notebook_utils import load_pits, create_ectp_slabs, hess_rcparams
from paper_figure_utils import save_paper_figure
from loss_analysis_utils import (
    analyze_losses, plot_histogram, comparison_table,
    plot_stepwise_losses, plot_stepwise_comparison, plot_loss_heatmap, plot_method_causes,
    successful_pathway_table, export_successful_pathway_table, TARGET_LABELS,
    plot_pathway_distribution,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 2)
warnings.filterwarnings("ignore", message="Using UFloat objects with std_dev==0.*", category=UserWarning)
hess_rcparams()
plt.rcParams["mathtext.fallback"] = "cm"  # Math arrows missing from Helvetica.
plt.rcParams["figure.dpi"] = 120  # Inline previews; exports retain 300 dpi.


def show_details(result, parameter, method=None, density_context=None):
    """Readable view; exact numeric bounds and counts remain in result.reasons."""
    frame = result.details(parameter, method, density_context)
    if frame.empty:
        print(f"{parameter}: no recorded failure causes.")
        return
    frame = frame.copy()
    frame["observed"] = [
        f"{low:g}–{high:g}" if pd.notna(low) else examples
        for low, high, examples in zip(
            frame.observed_min, frame.observed_max, frame.observed_examples
        )
    ]
    display(frame[["method", "density_context", "category", "reason", "field",
                   "grain_form", "resolved_grain_form", "slabs", "layers",
                   "observed", "interval", "unit"]])

from review_diagnostics import load_diagnostics
results_dir = repo_root / "paper_analysis/results/corrected"


## 1. Load the canonical cohort summaries

The runner parses sorted CAAML files and retains one slab per qualifying ECTP result. Multiple tests remain separate. The cached diagnostics preserve the raw-processing audit flags.


In [2]:
import json
summary = json.loads((results_dir / "summary.json").read_text())
print(f"Loaded canonical evidence for {summary['pits']:,} pits and {summary['slabs']:,} ECTP slabs.")


Loaded canonical evidence for 50,278 pits and 14,776 ECTP slabs.


### How to read the loss tables

A pathway succeeds only when all its required layer calculations succeed and its target output is valid: finite shear weight for either target (zero is allowed). For the elasticity target, every layer must also have successful Young’s modulus and Poisson’s-ratio calculations. Calculations propagate input measurement uncertainty only.

Figure labels follow the manuscript: density $\rho$, shear weight $W_{\mathrm{shear}}$, Young's modulus $E$, Poisson's ratio $\nu$, and slope angle $\psi$. Method labels use Geldsetzer, Kim T2, Kim T6, Bergfeld, Köchle, Wautier, Schöttner, and Srivastava. **Direct matched density** means density observations aligned to the stratigraphic layers. Grain form in the analysis means the observed **primary grain form**.

**Stepwise losses** use an explicit reporting order to assign each failed slab–pathway attempt to its first unsuccessful step. Each row reconciles `entering = surviving + newly_lost`. The newly lost slabs are partitioned into missing observations only, method validity limits only, both (mixed), or other errors. This order is a reporting convention for a branched dependency graph.

**Method causes** include all identifiable constraints, even on downstream branches. Missing upstream calculated density, E, or Poisson's ratio is labeled **blocked**, not a new missing observation. Density limits are checked only when the selected density method produces a finite value. An observed unsupported grain form is a validity failure; an absent grain form is missing data. Unknown hardness codes are recorded as other errors.

Cause counts overlap: a slab or layer can violate several constraints. `slabs` counts distinct ECTP slabs; `layers` counts distinct original layer observations, including a layer only once when it is shared by several slabs. Shared method steps are also counted once across pathways. `density_context` identifies the selected upstream density method for E and Srivastava; Köchle's Poisson's ratio is independent of density. Context counts must not be added to obtain a unique total.

The diagnostic tables follow the **implemented** domains. They impose no numerical density range on Schöttner and no grain-size range on Kim T6. Observed numeric ranges in the detail tables show the minimum and maximum violating values; validity intervals are in the displayed units. Empty intervals mean no numerical range applies. The complete detail table is available as `result.reasons`; use `show_details` to select a method or density context.


## 2. Calculate both targets

Calculate both analyses before displaying coverage tables or loss figures. All later sections use these results.

### Shear weight, $W_{\mathrm{shear}}$

Reporting order: density → shear weight. Thickness is always available in this cohort, so it does not need a separate reporting step. The calculation still checks layer thickness and integrated slab weight; any thickness failure in another dataset is attributed to the shear-weight requirement.


In [3]:
weight = load_diagnostics(results_dir, "slab_weight_shear")
assert weight.n_pathways == 4
print(f"{len(weight.attempts):,} slab–pathway attempts")


59,104 slab–pathway attempts


### Shear weight with elasticity

Reporting order: density → shear weight (layer thickness and slope angle) → Young's modulus → Poisson's ratio. This follows the paper's [shear-weight-with-elasticity attrition figure](../paper/figures/slab_weight_shear_with_elasticity_attrition.png). The methods and their domains are illustrated in [paper_diagrams.ipynb](../examples/paper_diagrams.ipynb).

The target retains the calculated shear weight once **both elasticity parameters succeed on every layer**; elasticity is a coverage requirement and does not change the weight value. No layer depth is required. The last stage also checks the final finite target output; any residual output error is classified as an other error there.

**Measured-density fallback remains removed from the dispatcher.** The explicitly selected direct-measurement pathway remains available. A failed empirical density step blocks density-dependent calculations even when measured density exists. All 32 combinations are recalculated using only input measurement uncertainty.

For Kim T2 → Wautier → Köchle, the paper figure retains **5,907 → 5,470 → 1,901 → 687 slabs** from the starting cohort of 14,776. The following assertion checks that exact sequence.


In [4]:
elastic_weight = load_diagnostics(results_dir, "slab_weight_shear_with_elasticity")
assert elastic_weight.n_pathways == 32
assert not elastic_weight.attempts.loc[~elastic_weight.attempts.density_ok, "success"].any()
paper_pathway = "Kim T2 → Wautier → Köchle"
paper_stages = elastic_weight.stages[elastic_weight.stages.pathway == paper_pathway].sort_values("step")
assert paper_stages.iloc[-1].surviving == int(elastic_weight.coverage.loc[elastic_weight.coverage.pathway == paper_pathway, "successful_slabs"].iloc[0])
print(f"{len(elastic_weight.attempts):,} attempts; {elastic_weight.attempts.success.sum():,} complete pathways succeeded.")
print(f"{elastic_weight.n_slabs - elastic_weight.histogram.loc[0]:,} slabs have at least one successful elasticity pathway.")


472,832 attempts; 7,971 complete pathways succeeded.
772 slabs have at least one successful elasticity pathway.


### Reconciliation checks

Both histograms include every starting slab; all pathway-step partitions reconcile with the calculated successes. The detailed missing/validity/blocked counts intentionally overlap and are not summed.

In [5]:
for result in [weight, elastic_weight]:
    assert result.histogram.sum() == 14_776
    assert len(result.attempts) == 14_776 * result.n_pathways
    assert result.stages.newly_lost.sum() + result.attempts.success.sum() == len(result.attempts)
assert len(weight.histogram) == 5 and len(elastic_weight.histogram) == 33
print("Cohort, histogram, pathway coverage, and stepwise-loss checks passed.")

Cohort, histogram, pathway coverage, and stepwise-loss checks passed.


### Audit of “Other errors”

“Other errors” is a residual first-failure category, not another name for missing observations or validity limits. Possible causes include an unrecognized hand-hardness code or an unexplained calculation/output failure. The method diagnostics independently record `other` issues across all inspected branches; these overlapping details must not be summed as additional failed attempts. `blocked` records describe consequences of unsuccessful upstream calculations.

The audit below distinguishes residual first-failure attempts from detailed diagnostic rows and lists any recorded other-error reasons. A zero count is an explicit result, not an omitted plotting category.


In [6]:
audit_rows = []
for result in [weight, elastic_weight]:
    other_details = result.reasons[result.reasons.category == "other"]
    audit_rows.append({
        "Target": TARGET_LABELS[result.target],
        "First-failure attempts: other errors": int((result.attempts.failure_category == "other").sum()),
        "Detailed other-error reason rows": len(other_details),
    })
    if other_details.empty:
        print(f"{TARGET_LABELS[result.target]}: no other errors in the method diagnostics.")
    else:
        display(other_details)
display(pd.DataFrame(audit_rows).set_index("Target"))


Shear weight: no other errors in the method diagnostics.
Shear weight with elasticity: no other errors in the method diagnostics.


                              First-failure attempts: other errors  \
Target                                                               
Shear weight                                                     0   
Shear weight with elasticity                                     0   

                              Detailed other-error reason rows  
Target                                                          
Shear weight                                                 0  
Shear weight with elasticity                                 0  

## 3. Compare coverage

The first three count columns describe distinct slabs. Successful attempts and lost-attempt categories count slab–pathway pairs; shear weight with elasticity offers eight times as many pathways as shear weight alone. The loss-category columns assign each attempt to its first unsuccessful step and sum to total failed attempts.


In [7]:
display(comparison_table(weight, elastic_weight).T)


target                            slab_weight_shear  \
pathways                                          4   
zero successful pathways                       9241   
at least one successful pathway                5535   
all pathways successful                          20   
successful slab–pathway attempts              10463   
lost attempts: missing_only                   28355   
lost attempts: validity_only                   7562   
lost attempts: mixed                          12724   
lost attempts: other                              0   

target                            slab_weight_shear_with_elasticity  
pathways                                                         32  
zero successful pathways                                      14004  
at least one successful pathway                                 772  
all pathways successful                                           0  
successful slab–pathway attempts                               7971  
lost attempts: missing_only  

In [8]:
display(weight.coverage[["pathway", "successful_slabs", "coverage_percent"]])
display(elastic_weight.coverage[["pathway", "successful_slabs", "coverage_percent"]])


                  pathway  successful_slabs  coverage_percent
0                  Kim T2              5470             37.02
1              Geldsetzer              4187             28.34
2                  Kim T6               697              4.72
3  Direct matched density               109              0.74

                                            pathway  successful_slabs  \
0                         Kim T2 → Wautier → Köchle               687   
1                     Geldsetzer → Wautier → Köchle               684   
2                      Geldsetzer → Köchle → Köchle               677   
3                   Geldsetzer → Schöttner → Köchle               647   
4                          Kim T2 → Köchle → Köchle               581   
5                       Kim T2 → Schöttner → Köchle               580   
6                        Kim T2 → Bergfeld → Köchle               465   
7                    Geldsetzer → Bergfeld → Köchle               443   
8                     Kim T2 → Wautier → Srivastava               420   
9                 Geldsetzer → Wautier → Srivastava               418   
10                  Kim T2 → Schöttner → Srivastava               415   
11              Geldsetzer → Schöttner → Srivastava               393   
12                   Kim T2 → Bergfeld → Srivastava

### Distribution of successful calculation pathways

The paper table compares exact slab counts and percentages for both targets, including slabs with **zero successful pathways**. Percentages use all 14,776 ECTP slabs. A dash indicates a pathway count above the four available shear-weight pathways. The table and CSV show every integer count from 0 through 32, including rows with zero slabs.

The histograms below remain notebook diagnostics. Each shows every possible integer bin, including zero and unoccupied bins.

For a standalone PDF, run `latexmk -pdf successful_pathway_distribution_preview.tex` from either paper table directory. The un-suffixed `.tex` file is a manuscript include.


In [9]:
distribution = successful_pathway_table(weight, elastic_weight)
display(distribution.style.format({
    "Shear weight: slabs": "{:,.0f}", "With elasticity: slabs": "{:,.0f}",
    "Shear weight: %": "{:.2f}", "With elasticity: %": "{:.2f}",
}, na_rep="—"))
table_paths = export_successful_pathway_table(weight, elastic_weight, [
    repo_root / "paper/tables",
])
print("Exported full LaTeX table, standalone preview source, and complete-bin CSV to the repository paper table directory.")


Exported full LaTeX table, standalone preview source, and complete-bin CSV to the repository paper table directory.


### Successful-pathway counts and distributions

Each mechanical target has its own panel: an exact count/percentage table on the left, and a horizontal bar for the same row on the right. Both panels show all counts from 0 through 32 and share a linear slab-count scale. The row for zero successful pathways is included. Counts above four are inapplicable to shear weight and are shown as dashes with no bar. Percentages use all 14,776 ECTP slabs.


In [10]:
fig = plot_pathway_distribution(weight, elastic_weight)
saved = save_paper_figure(fig, "successful_pathway_distribution")
plt.show()
print(f"Exported PDF and PNG to {saved['repo_pdf'].parent}.")


<Figure size 840x984 with 4 Axes>

Exported PDF and PNG to /Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures.


In [11]:
plot_histogram(weight)
plt.show()
display(weight.histogram.to_frame().T)

<Figure size 720x456 with 1 Axes>

successful_pathways     0     1     2    3   4
slab_count           9241  1304  3554  657  20

In [12]:
plot_histogram(elastic_weight)
plt.show()
display(elastic_weight.histogram.to_frame())

<Figure size 1080x456 with 1 Axes>

                     slab_count
successful_pathways            
0                         14004
1                             2
2                            18
3                            16
4                            39
5                             4
6                           243
7                             1
8                            85
9                             9
10                            3
11                            0
12                           52
13                            0
14                            0
15                            1
16                          291
17                            0
18                            5
19                            0
20                            1
21                            0
22                            0
23                            0
24                            2
25                            0
26                            0
27                            0
28                            0
29      

## 4. Compare stepwise losses

Each pathway starts with all 14,776 ECTP slabs. Its final surviving count equals its successful target count. Each calculation bar shows the number entering that requirement; surviving slabs and mutually exclusive newly lost categories sum to its width. In the elasticity panel, a final target bar repeats the successful output count with zero additional losses; it is a display step, not an extra calculation requirement.

Light gray denotes surviving slabs, muted teal missing observations only, and terracotta validity limits only. Mixed causes combine the teal fill with terracotta diagonal hatching. These colors describe loss causes, independently of the method colors elsewhere in the paper. Other errors are shown in charcoal whenever present.

Shear weight reports **density → shear weight**. Shear weight with elasticity reports **density → shear weight → Young's modulus → Poisson's ratio**. Both shear-weight requirements check layer thickness and slope angle; the separate thickness/integrated-weight step is omitted because thickness is present throughout this cohort. This reporting order describes attrition through a branched dependency graph.

Calculation parameters label the y-axis. Method names appear only in the title, each followed by the parameter it calculates in parentheses. Parenthetical axis details describing joined inputs use **Join:**. The x-axis shows ECTP slab counts. The elasticity panel’s final bar names the target parameter and shows its final count with no losses; the shear-weight panel ends at its shear-weight calculation. Defaults show the highest-coverage Kim T2 and Kim T2 → Wautier → Köchle pathways. Change the selections to any label in the corresponding coverage table.

The two targets appear side by side with the same slab-count scale and bar height, plus a shared legend. The shear-weight panel has no duplicate final-target bar and leaves the lower elasticity-only rows blank. Standalone figure files are also retained.

Percentages beside loss counts use the original 14,776 starting slabs as the denominator at every step, including the final zero-loss bar.


In [13]:
WEIGHT_PATHWAY = "Kim T2"  # Direct matched density, Geldsetzer, Kim T2, or Kim T6
ELASTICITY_PATHWAY = "Kim T2 → Wautier → Köchle"

# Retain standalone exports alongside the combined comparison.
for result, pathway, stem in [
    (weight, WEIGHT_PATHWAY, "loss_shear_weight_stepwise"),
    (elastic_weight, ELASTICITY_PATHWAY, "loss_elasticity_stepwise"),
]:
    single = plot_stepwise_losses(result, pathway)
    save_paper_figure(single, stem, close=True)

fig = plot_stepwise_comparison(weight, elastic_weight, WEIGHT_PATHWAY, ELASTICITY_PATHWAY)
saved = save_paper_figure(fig, "loss_stepwise_comparison")
plt.show()
print(f"Exported combined PDF and PNG to {saved['repo_pdf'].parent}.")


<Figure size 840x624 with 2 Axes>

Exported combined PDF and PNG to /Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures.


### Stepwise loss tables

The tables in `weight.stages` and `elastic_weight.stages` contain all reporting steps for every pathway. Uncomment the displays below to inspect their exact counts.


In [14]:
# display(weight.stages)

In [15]:
#display(elastic_weight.stages)

### Compare stepwise losses across pathways

The heatmaps for both targets appear together below. Each cell shows the percentage of the **original 14,776 slabs** newly lost at that step. Rows are pathways; columns are requirements. Elasticity panels are grouped by density method, with E → Poisson's-ratio combinations as rows. All panels use the same 0–100% color scale. Values below 0.1% are marked `<0.1`; a zero means no slabs were newly lost.


In [16]:
plot_loss_heatmap(weight)
plt.show()

<Figure size 900x456 with 2 Axes>

In [17]:
plot_loss_heatmap(elastic_weight)
plt.show()

<Figure size 1560x1080 with 5 Axes>

## 5. Method causes and details

The summary deduplicates affected slabs and layers within each method, upstream context, and cause category. The detail table separates unsupported grain codes, grain-specific hardness bounds, and missing density, hardness, grain form, or grain size. Slope loss is recorded at slab level, so its layer count is zero.

Density details are shared by both targets because the same four density methods act on the same cohort. The density detail view initially shows Kim T2, the highest-coverage method. Change `DENSITY_METHOD` below to inspect Geldsetzer, Kim T6, or direct measurement; use `None` to show all density methods. Uncomment the displays to inspect the tables.


In [18]:
#display(weight.method_summary)
DENSITY_METHOD = "kim_jamieson_table2"  # data_flow, geldsetzer, kim_jamieson_table6, or None
#show_details(weight, "density", DENSITY_METHOD)
#show_details(weight, "slab_weight")
#show_details(weight, "slab_weight_shear")

### Detailed density-method causes

The teal and terracotta panels show missing observations and method validity limits for the selected density method. These counts **overlap**: a slab can appear in several bars. Each panel shows its 12 largest detailed causes; `weight.details("density", DENSITY_METHOD)` retains all causes. Missing hand hardness is shown as one bar across primary grain forms, counting each affected slab once. Other grain-specific causes remain separate, and numerical labels include the method's bound. Blocked calculations and other errors remain in the tables.


In [19]:
if DENSITY_METHOD is not None:
    plot_method_causes(weight, "density", DENSITY_METHOD, top_n=12)
    plt.show()
else:
    print("Select one DENSITY_METHOD above to display its cause bars.")

<Figure size 1320x1581.6 with 2 Axes>

### Elasticity-method causes

The elasticity summary includes every method and density context with a recorded cause. Unsupported downstream grain forms remain identifiable even if density could not be calculated; an unavailable density is listed separately as blocked. Uncomment the display below to inspect the summary.


In [20]:
#display(elastic_weight.method_summary)

### Grain form, hardness, and density-range details

Below, choose any E method to inspect its grain-specific failures and the density values outside its validity range. The full table for all E methods is `elastic_weight.details("elastic_modulus")`. Poisson's-ratio details show all methods and density contexts. Uncomment the displays to inspect the tables. The E detail view initially uses Kim T2 density; set `DENSITY_CONTEXT = None` for all density methods.

In [21]:
E_METHOD = "wautier"  # bergfeld, kochle, wautier, or schottner
DENSITY_CONTEXT = "kim_jamieson_table2"  # None for all, or another density method
#show_details(elastic_weight, "elastic_modulus", E_METHOD, DENSITY_CONTEXT)
#show_details(elastic_weight, "poissons_ratio")
#show_details(elastic_weight, "slab_weight")
#show_details(elastic_weight, "slab_weight_shear")
#show_details(elastic_weight, "slab_weight_shear_with_elasticity")

### Detailed elasticity-method causes

The E chart uses `E_METHOD` and `DENSITY_CONTEXT` from the detail view, initially matching the paper's Wautier and Kim T2 pathway. The Poisson's-ratio chart initially shows Köchle. Choose `NU_METHOD = "srivastava"` to inspect its density threshold as well as grain-form constraints, using the selected density context.

Each panel shows the 12 largest overlapping detailed causes. Raw grain subclasses are not combined, and context counts are not added together. The full counts and intervals remain in `elastic_weight.reasons` and the detail tables.

In [22]:
plot_method_causes(elastic_weight, "elastic_modulus", E_METHOD, DENSITY_CONTEXT, top_n=12)
plt.show()

NU_METHOD = "kochle"  # kochle or srivastava
nu_context = DENSITY_CONTEXT if NU_METHOD == "srivastava" else None
plot_method_causes(elastic_weight, "poissons_ratio", NU_METHOD, nu_context, top_n=12)
plt.show()

<Figure size 1320x1507.2 with 2 Axes>

<Figure size 1320x1507.2 with 2 Axes>

### Compare bottlenecks

The bottleneck table lists the three largest missing-observation and method-validity counts for each target. These are overlapping affected-slab counts within method contexts, not expected numbers of recoverable slabs. Blocked downstream steps are consequences of upstream loss and are excluded from this ranking. Uncomment the display below to inspect the table.


In [23]:
bottleneck_rows = []
for result in [weight, elastic_weight]:
    for category in ["missing", "validity"]:
        largest = (result.method_summary[result.method_summary.category == category]
                   .sort_values(["slabs", "parameter", "method", "density_context"],
                                ascending=[False, True, True, True]).head(3).copy())
        largest.insert(0, "target", result.target)
        bottleneck_rows.append(largest)
# display(pd.concat(bottleneck_rows, ignore_index=True))